In [1]:
import pandas as pd
import numpy as np
import joblib

from xgboost import XGBRegressor

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder
)

from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    RandomizedSearchCV
)

from sklearn.metrics import r2_score

In [2]:
df = pd.read_csv("recidivism_score.csv")

df.head()

,Age,Gender,Sentence Type,education_level,Sentence_duration_months,Incident_count,violent_incidents_count,escape_incidents_count,Recidivism_score
0,58,Male,Kidnapping,Primary,155,12,1,2,70.0
1,65,Male,Kidnapping,Secondary,271,8,2,1,55.8
2,45,Male,Theft,Primary,41,16,3,4,83.3
3,19,Male,Sexual Offense,Secondary,203,10,3,3,94.3
4,32,Male,Robbery,Bachelor's,133,9,2,1,54.4


In [3]:
X = df.drop(
    "Recidivism_score",
    axis=1
)

y = df["Recidivism_score"]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42
)

In [5]:
numeric_features = [

    "Age",

    "Sentence_duration_months",

    "Incident_count",

    "violent_incidents_count",

    "escape_incidents_count"

]

ordinal_features = [

    "education_level"

]

categorical_features = [

    "Gender",

    "Sentence Type"

]

In [6]:
education_order = [[

    "Illiterate",

    "Literate",

    "Primary",

    "Preparatory",

    "Secondary",

    "Bachelor's",

    "Postgraduate education"

]]

In [7]:
ordinal_pipeline = Pipeline([

    (
        "ordinal",
        OrdinalEncoder(
            categories=education_order
        )
    )

])

categorical_pipeline = Pipeline([

    (
        "onehot",

        OneHotEncoder(

            drop="first",

            handle_unknown="ignore"

        )
    )

])

In [8]:
preprocessor = ColumnTransformer([

    (

        "ordinal",

        ordinal_pipeline,

        ordinal_features

    ),

    (

        "categorical",

        categorical_pipeline,

        categorical_features

    )

],
remainder="passthrough")

In [9]:
pipeline = Pipeline([

    (

        "preprocessor",

        preprocessor

    ),

    (

        "model",

        XGBRegressor(

            random_state=42,

            objective="reg:squarederror"

        )

    )

])

In [10]:
cv_model = Pipeline([

    (

        "preprocessor",

        preprocessor

    ),

    (

        "model",

        XGBRegressor(

            n_estimators=500,

            learning_rate=0.05,

            max_depth=5,

            subsample=0.8,

            colsample_bytree=0.8,

            random_state=42

        )

    )

])

In [12]:
scores = cross_val_score(

    cv_model,

    X,

    y,

    cv=5,

    scoring="r2",

    n_jobs=-1

)

for i, score in enumerate(scores, start=1):

    print(f"Fold {i}: {score:.4f}")

print("--------------------------------")

print(f"Average R² : {scores.mean():.4f}")

print(f"St.D        : {scores.std():.4f}")

print(f"Best Fold  : {scores.max():.4f}")

print(f"Worst Fold : {scores.min():.4f}")

Fold 1: 0.9391
Fold 2: 0.9412
Fold 3: 0.9412
Fold 4: 0.9421
Fold 5: 0.9359
--------------------------------
Average R² : 0.9399
St.D        : 0.0022
Best Fold  : 0.9421
Worst Fold : 0.9359


In [17]:
param_dist = {

    "model__n_estimators":[200,300,500,700,1000],

    "model__learning_rate":[0.01,0.03,0.05,0.1,0.2],

    "model__max_depth":[3,4,5,6,7],

    "model__subsample":[0.6,0.8,1.0],

    "model__colsample_bytree":[0.6,0.8,1.0],

    "model__min_child_weight":[1,3,5],

    "model__gamma":[0,0.1,0.2,0.3]

}
random_search = RandomizedSearchCV(

    estimator=pipeline,

    param_distributions=param_dist,

    n_iter=20,

    scoring="r2",

    cv=5,

    random_state=42,

    n_jobs=-1

)
random_search.fit(
    X_train,
    y_train
)
print(random_search.best_params_)

{'model__subsample': 0.8, 'model__n_estimators': 700, 'model__min_child_weight': 3, 'model__max_depth': 4, 'model__learning_rate': 0.05, 'model__gamma': 0.2, 'model__colsample_bytree': 0.8}


In [19]:
best_model = random_search.best_estimator_
y_pred = best_model.predict(
    X_test
)
r2 = r2_score(
    y_test,
    y_pred
)

print(r2)

0.9421917956586974


In [20]:
joblib.dump(

    best_model,

    "recidivism_score_pipeline.pkl"

)

['recidivism_score_pipeline.pkl']

In [21]:
recidivism_pipeline = joblib.load(

    "recidivism_score_pipeline.pkl"

)

In [22]:
new_data = pd.DataFrame({

    "Age":[25],

    "Gender":["Male"],

    "Sentence Type":["Theft"],

    "education_level":["Secondary"],

    "Sentence_duration_months":[36],

    "Incident_count":[10],

    "violent_incidents_count":[2],

    "escape_incidents_count":[1]

})

prediction = recidivism_pipeline.predict(new_data)

print(prediction)

[46.241795]


In [23]:
new_data = pd.DataFrame({

    "Age":[60],

    "Gender":["Male"],

    "Sentence Type":["Public Order"],

    "education_level":["Bachelor's"],

    "Sentence_duration_months":[6],

    "Incident_count":[2],

    "violent_incidents_count":[0],

    "escape_incidents_count":[0]

})

prediction = recidivism_pipeline.predict(new_data)

print(prediction)

[5.350933]


In [24]:
new_data = pd.DataFrame({

    "Age":[35],

    "Gender":["Male"],

    "Sentence Type":["Fraud"],

    "education_level":["Preparatory"],

    "Sentence_duration_months":[48],

    "Incident_count":[8],

    "violent_incidents_count":[1],

    "escape_incidents_count":[1]

})

prediction = recidivism_pipeline.predict(new_data)

print(prediction)

[41.398216]
